In [1]:
import pandas as pd
import io
import re

In [2]:
path = '../data/raw/aves_eua_filtrado.csv'  #modificar o caminho, tipo de dado modificado

# resolve quebras de linha e problemas de encoding
with open(path, 'rb') as f:
    content = f.read().decode('utf-8', errors='ignore')
    content = content.replace('\r\n', '\n').replace('\r', '\n')

df = pd.read_csv(io.StringIO(content))

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/aves_eua_filtrado.csv'

In [ ]:
# limpeza 
def clean_text(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'[^\x00-\x7F]+', ' ', text) 
    text = " ".join(text.split())
    return text.lower()

df['description_clean'] = df['description'].apply(clean_text)


In [ ]:
print(f"\nestatísticas")
print(f"espécies únicas: {df['species'].nunique()}")
print(f"países: {df['country'].unique()}")
print(f"tamanho médio: {df['description_clean'].str.len().mean():.2f} caracteres")

display(df[['species', 'description_clean']].head(10))


estatísticas
espécies únicas: 425
países: ['United States of America']
tamanho médio: 213.18 caracteres


,species,description_clean
0,Gavia immer,dc birds exhibit; catalog number verified from...
1,Setophaga fusca,ahy male by plumage (idenitical in plumage & r...
2,Podiceps nigricollis,"tag also says ""american museum of natural hist..."
3,Colinus virginianus,"tag notes ""[237 (j.)]"", ""ortyx virginianus"", ""..."
4,Aphelocoma woodhouseii,"*[""july 7"" on sennett card & label; ""9/20"" on ..."
5,Sternula antillarum,"""note the jet black bill & the peculiar state ..."
6,Dryobates pubescens,"locality: new york; erie co., boston collected..."
7,Actitis macularius,birds of dc exhibit; catalog number verified f...
8,Dryobates pubescens,"locality: florida; volusia co., south daytona,..."
9,Podiceps nigricollis,"ledger says this was collected in the ""spring ..."


In [ ]:
df.head()

,species,family,description,scientificName,country,description_clean
0,Gavia immer,Gaviidae,DC Birds Exhibit; catalog number verified from...,"Gavia immer (Brunnich, 1764)",United States of America,dc birds exhibit; catalog number verified from...
1,Setophaga fusca,Parulidae,AHY MALE BY PLUMAGE (IDENITICAL IN PLUMAGE & R...,"Dendroica fusca (Statius Müller, 1776)",United States of America,ahy male by plumage (idenitical in plumage & r...
2,Podiceps nigricollis,Podicipedidae,"Tag also says ""American Museum of Natural Hist...","Podiceps nigricollis C.L.Brehm, 1831",United States of America,"tag also says ""american museum of natural hist..."
3,Colinus virginianus,Odontophoridae,"Tag notes ""[237 (J.)]"", ""Ortyx virginianus"", ""...","Colinus virginianus (Linnaeus, 1758)",United States of America,"tag notes ""[237 (j.)]"", ""ortyx virginianus"", ""..."
4,Aphelocoma woodhouseii,Corvidae,"*[""July 7"" on Sennett card & label; ""9/20"" on ...","Aphelocoma woodhouseii (S.F.Baird, 1858)",United States of America,"*[""july 7"" on sennett card & label; ""9/20"" on ..."


In [ ]:


print("valores únicos")
cols = ['species', 'family', 'scientificName', 'country']
for col in cols:
    print(f"{col:15}: {df[col].nunique()} valores")


print("\nteste de normalização")
for col in cols:
    original = df[col].nunique()
    normalized = df[col].astype(str).str.lower().str.strip().nunique()
    if original != normalized:
        print(f" {col}: têm {original - normalized} valores quase iguais")
    else:
        print(f" {col}: limpo")

valores únicos
species        : 425 valores
family         : 70 valores
scientificName : 543 valores
country        : 1 valores

teste de normalização
 species: limpo
 family: limpo
 scientificName: limpo
 country: limpo


In [ ]:
# Comparando os 5 primeiros nomes científicos de uma mesma espécie
ex_species = df['species'].iloc[0]
print(f"\nvariações pra espécie: {ex_species}")
print(df[df['species'] == ex_species]['scientificName'].unique())


variações pra espécie: Gavia immer
['Gavia immer (Brunnich, 1764)']


In [ ]:
df['desc_prefix'] = df['description'].str[:50]
duplicatas_prefixo = df['desc_prefix'].duplicated().sum()

print(f"\n análise de descrições")
print(f" idênticos: {duplicatas_prefixo}")



 análise de descrições
 idênticos: 214


In [ ]:
print("amostra de inícios idênticos")
print(df[df['desc_prefix'].duplicated(keep=False)].sort_values('desc_prefix')['description'].head(10).values)
df_clean = df.drop_duplicates(subset=['desc_prefix'], keep='first').copy()

df_clean['species'] = df_clean['species'].astype(str).str.lower().str.strip()
df_clean['description_final'] = df_clean['description'].astype(str).str.lower().str.strip()

print(f"Registros restantes: {len(df_clean)}")
print(f"Espécies únicas: {df_clean['species'].nunique()}")

amostra de inícios idênticos
['AD.  BILL FROM ANT. NARES TO TIP 18.6 MM, BILL DEPTH @ ANT. NARES 8.9 MM, BILL WIDTH @ ANT. NARES 8.9 MM, CULMEN 25.5 MM, TARSUS 38.8 MM, HALLUX 12.4 MM, WING CHORD 108.8 MM, TAIL 129.4 MM; MEASUREMENTS BY GLEN E. WOOLFENDEN USING METHOD OF PITELKA 1951.'
 'AD.  BILL FROM ANT. NARES TO TIP 18.6 MM, BILL DEPTH @ ANT. NARES 8.7 MM, BILL WIDTH @ ANT. NARES 9.0 MM, CULMEN 25.9 MM, TARSUS 38.3 MM, HALLUX 12.9 MM, WING CHORD 117.2 MM, TAIL 139.3 MM; MEASUREMENTS BY GLEN E. WOOLFENDEN USING METHOD OF PITELKA 1951.'
 'AD. BILL FROM ANT. NARES TO TIP 17.8 MM, BILL DEPTH @ ANT. NARES 9.4 MM, BILL WIDTH @ ANT. NARES 10.5 MM, CULMEN 26.9, TARSUS 38.6, HALLUX 13.0, WING CHORD 119.7, TAIL 147.6 MM; MEASUREMENTS BY GLEN E. WOOLFENDEN USING METHOD OF PITELKA 1951.'
 'AD. BILL FROM ANT. NARES TO TIP 17.8 MM, BILL DEPTH @ ANT. NARES 9.3 MM, BILL WIDTH @ ANT. NARES 10.0 MM, CULMEN 26.0 MM, TARSUS 36.6 MM, HALLUX 11.8 MM, WING CHORD 110.0 MM, TAIL 127.7 MM; MEASUREMENTS BY G

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# filtro de frequência para reduzir ruído e dimensionalidade
# max_df=0.8 -> ignora palavras que aparecem em mais de 80% das descrições
# min_df=2   -> ignora palavras que aparecem em apenas 1 registro
vectorizer = TfidfVectorizer(
    stop_words='english', 
    max_df=0.8, 
    min_df=2,
    ngram_range=(1, 2) 
)

tfidf_matrix = vectorizer.fit_transform(df_clean['description_final'])

print(f"dimensões da matriz: {tfidf_matrix.shape}")
print(f"vocabulário total: {len(vectorizer.get_feature_names_out())} termos técnicos.")

dimensões da matriz: (1371, 5900)
vocabulário total: 5900 termos técnicos.


In [ ]:
import spacy

# Carrega o modelo de inglês
nlp = spacy.load("en_core_web_sm")

def super_cleaner(text):
    # Processa o texto com spaCy
    doc = nlp(text.lower())
    
    # Lista de entidades para remover (Pessoas, Organizações, Datas)
    to_remove = ['PERSON', 'ORG', 'DATE', 'GPE']
    ents_to_delete = [ent.text.lower() for ent in doc.ents if ent.label_ in to_remove]
    
    clean_tokens = []
    for token in doc:
        # Filtros: 
        # 1. Não é stop word
        # 2. É uma palavra alfabética
        # 3. Não está na lista de entidade
        if not token.is_stop and token.is_alpha and token.text not in ents_to_delete:
            clean_tokens.append(token.lemma_) # Pega a raiz da palavra
            
    return " ".join(clean_tokens)

df_clean['description_ultra'] = df_clean['description_final'].apply(super_cleaner)
print("texto gerado")

texto gerado


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_df=0.7,  # Palavras que aparecem em >70% das aves são ignoradas
    min_df=3,    # Palavras muito raras (erros) são ignoradas
    ngram_range=(1, 2) # Captura "blue wing" como uma unidade
)

tfidf_matrix = vectorizer.fit_transform(df_clean['description_ultra'])

print(f"dimensao pós mudança:{tfidf_matrix.shape}")
# Provavelmente o número de colunas diminuiu, o que é ÓTIMO (menos ruído)

dimensao pós mudança:(1371, 2477)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# calcula a matriz de distância entre todas as aves 

cos_sim = cosine_similarity(tfidf_matrix)

def buscar_aves_parecidas(index_da_ave, top_n=3):
    ave_alvo = df_clean.iloc[index_da_ave]
    print(f"\nave selecionada: {ave_alvo['species'].upper()}")
    print(f"descrição: {ave_alvo['description'][:150]}...")
    
    indices_similares = cos_sim[index_da_ave].argsort()[-(top_n+1):-1][::-1]
    
    print(f"\n aves mais parecidas (Morfologicamente):")
    return df_clean.iloc[indices_similares][['species', 'family', 'country']]


buscar_aves_parecidas(100)


ave selecionada: FALCO SPARVERIUS
descrição: Locality: New York; Erie Co., Buffalo, Buffalo Zoo Collection: 22 November by Dr. Crandall Data tag says "Falco s. sparverius" on back Also called a "...

 aves mais parecidas (Morfologicamente):


,species,family,country
81,falco sparverius,Falconidae,United States of America
72,falco sparverius,Falconidae,United States of America
1303,falco sparverius,Falconidae,United States of America


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def query_bird_model(user_input, top_n=3):
    # 1. Aplicar o tratamento (Lemmatization) no input do usuário
    # Usamos a mesma função 'super_cleaner' que criamos antes
    clean_input = super_cleaner(user_input)
    
    # 2. Transformar o input em um vetor TF-IDF
    # IMPORTANTE: Use .transform(), NÃO .fit_transform()
    # Queremos projetar no espaço já existente, não criar um novo
    query_vector = vectorizer.transform([clean_input])
    
    # 3. Calcular a similaridade com toda a base
    similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()
    
    # 4. Pegar os melhores resultados
    best_indices = similarities.argsort()[-top_n:][::-1]
    
    print(f"\n🔎 Resultado da busca para: '{user_input}'")
    print("-" * 50)
    
    results = []
    for idx in best_indices:
        score = similarities[idx]
        if score > 0: # Só mostra se houver alguma palavra em comum
            bird = df_clean.iloc[idx]
            results.append({
                "Espécie": bird['species'],
                "Confiança (Cosseno)": f"{score:.4f}",
                "Trecho da Descrição": bird['description'][:150] + "..."
            })
    
    if not results:
        print("❌ Nenhuma ave encontrada com esses termos.")
    else:
        return pd.DataFrame(results)

# --- TESTE AGORA ---
# Digite algo como: "small green bird with yellow belly" ou "large raptor with hooked bill"
texto_teste = input("Descreva as características da ave (em inglês): ")
query_bird_model(texto_teste)


🔎 Resultado da busca para: 'bird'
--------------------------------------------------


,Espécie,Confiança (Cosseno),Trecho da Descrição
0,buteo lineatus,0.3944,Observed here is a Buteo which is a relatively...
1,geococcyx californianus,0.3917,Individuals Nos. 2 and 3 recorded today; bill ...
2,podiceps auritus,0.3568,Birds of DC exhibit; catalog number label NOT ...


outras tentativas

In [ ]:
%pip install rank_bm25 sentence-transformers spacy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from rank_bm25 import BM25Okapi
import numpy as np
import pandas as pd

def build_bm25_model(df):
    print("índice BM25")
    # O BM25 precisa de listas de tokens (palavras separadas)
    tokenized_corpus = [doc.split(" ") for doc in df['description_ultra']]
    bm25 = BM25Okapi(tokenized_corpus)
    return bm25

def search_bm25(query, bm25_model, df, top_n=3):
    tokenized_query = query.lower().split(" ")
    # O BM25 já calcula os scores diretamente
    doc_scores = bm25_model.get_scores(tokenized_query)
    
    # Pega os índices dos maiores scores
    top_indices = np.argsort(doc_scores)[::-1][:top_n]
    
    results = []
    for idx in top_indices:
        if doc_scores[idx] > 0:
            results.append({
                "Score BM25": f"{doc_scores[idx]:.2f}",
                "Espécie": df.iloc[idx]['species'],
                "Descrição": df.iloc[idx]['description'][:150] + "..."
            })
    return pd.DataFrame(results)


bm25_index = build_bm25_model(df_clean)
search_bm25("large raptor hooked bill", bm25_index, df_clean)

índice BM25


,Score BM25,Espécie,Descrição
0,9.35,circus cyaneus,Temperate afternoon. Flying over Paynes prairi...
1,7.75,phalacrocorax auritus,Observed preening itself in the warm rain on t...
2,6.31,buteo lineatus,conditions: mostly cloudy location: residentia...


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

def build_lsa_model(df, n_components=100):
    print(f"espaço semântico LSA ({n_components} dimensões)...")
    
    vectorizer = TfidfVectorizer(max_df=0.8, min_df=2, ngram_range=(1, 2))
    tfidf_matrix = vectorizer.fit_transform(df['description_ultra'])
    
    # Aplica o SVD para compressão dimensional
    svd_model = TruncatedSVD(n_components=n_components, random_state=42)
    lsa_matrix = svd_model.fit_transform(tfidf_matrix)
    
    return vectorizer, svd_model, lsa_matrix

def search_lsa(query, vectorizer, svd_model, lsa_matrix, df, top_n=3):
    # Transforma a query no espaço TF-IDF e depois comprime pelo SVD
    query_tfidf = vectorizer.transform([query])
    query_lsa = svd_model.transform(query_tfidf)
    
    # Calcula similaridade no espaço comprimido (muito mais rápido e semântico)
    similarities = cosine_similarity(query_lsa, lsa_matrix).flatten()
    best_indices = similarities.argsort()[-top_n:][::-1]
    
    results = []
    for idx in best_indices:
        results.append({
            "Similaridade LSA": f"{similarities[idx]:.4f}",
            "Espécie": df.iloc[idx]['species'],
            "Descrição": df.iloc[idx]['description'][:150] + "..."
        })
    return pd.DataFrame(results)


vec, svd, lsa_mat = build_lsa_model(df_clean)
search_lsa("large raptor hooked bill", vec, svd, lsa_mat, df_clean)

espaço semântico LSA (100 dimensões)...


,Similaridade LSA,Espécie,Descrição
0,0.6018,phalacrocorax auritus,Observed preening itself in the warm rain on t...
1,0.5353,charadrius semipalmatus,This bird has a black bill and the bill is lar...
2,0.5330,anhinga anhinga,"The darters, anhingas, or snakebirds are mainl..."


In [ ]:
from sentence_transformers import SentenceTransformer, util
import torch
import pandas as pd

def build_sbert_model(df):
    print("modelo SBERT e embeddings...")
    # 'all-MiniLM-L6-v2' é um modelo extremamente rápido e leve
    model = SentenceTransformer('all-MiniLM-L6-v2')
    
    # Codifica todo o corpus para vetores densos (Tensores do PyTorch)
    # Usamos a descrição original limpa (sem lemmatization pesada), 
    # pois o modelo precisa da gramática para entender o contexto.
    corpus_embeddings = model.encode(df['description_final'].tolist(), convert_to_tensor=True)
    
    return model, corpus_embeddings

def search_sbert(query, model, corpus_embeddings, df, top_n=3):
    # Codifica a query no mesmo espaço tensorial
    query_embedding = model.encode(query, convert_to_tensor=True)
    
    # O PyTorch calcula a similaridade do cosseno via GPU/CPU de forma ultra rápida
    cos_scores = util.cos_sim(query_embedding, corpus_embeddings)[0]
    
    # Extrai os melhores resultados
    top_results = torch.topk(cos_scores, k=top_n)
    
    results = []
    for score, idx in zip(top_results[0], top_results[1]):
        results.append({
            "Score Semântico": f"{score.item():.4f}",
            "Espécie": df.iloc[idx.item()]['species'],
            "Descrição": df.iloc[idx.item()]['description'][:150] + "..."
        })
    return pd.DataFrame(results)

# --- Uso ---
sbert_model, embeddings = build_sbert_model(df_clean)
search_sbert("bird of prey with curved mouth", sbert_model, embeddings, df_clean)

modelo SBERT e embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,Score Semântico,Espécie,Descrição
0,0.5803,pandion haliaetus,It was around 5 p.m. with temperatures around ...
1,0.5586,cygnus buccinator,Three birds foraging actively. ID based on rel...
2,0.5454,sialia sialis,In the morning I was walking home from depot p...


In [ ]:


# O BM25 precisa do corpus tokenizado (lista de listas de palavras)
tokenized_corpus = [doc.split(" ") for doc in df_clean['description_ultra']]
bm25_model = BM25Okapi(tokenized_corpus)


from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

vectorizer_lsa = TfidfVectorizer(max_df=0.8, min_df=2, ngram_range=(1, 2))
tfidf_mat = vectorizer_lsa.fit_transform(df_clean['description_ultra'])


svd_model = TruncatedSVD(n_components=100, random_state=42)
lsa_matrix = svd_model.fit_transform(tfidf_mat)


print("🧠 Construindo SBERT (Isso pode demorar um pouco na primeira vez)...")
from sentence_transformers import SentenceTransformer
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
corpus_embeddings = sbert_model.encode(df_clean['description_final'].tolist(), convert_to_tensor=True)

print("\n ok")

🧠 Construindo SBERT (Isso pode demorar um pouco na primeira vez)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



 ok


In [ ]:
df_detalhado, df_resumo = evaluate_models(
    df_clean, 
    bm25_model, 
    vectorizer_lsa, 
    svd_model, 
    lsa_matrix, 
    sbert_model, 
    corpus_embeddings
)

print("\nresultados")
display(df_resumo)

🔬 Iniciando Benchmarking (Top-3 Evaluation)...


resultados


,Model,Mean Precision@K,Avg Latency (ms)
0,BM25 (Lexical),0.083333,2.727850
1,LSA (SVD/Linear),0.083333,5.677175
2,SBERT (Neural),0.000000,13.593025
